# 02 - Data Preparation

This notebook prepares the Helpdesk Tickets dataset for analysis. It loads the dataset created/used during data understanding, checks data quality, handles missing values and duplicates, creates new features, and saves a cleaned dataset for the next steps.

## 1. Load Libraries and Dataset

We import the required libraries and load the dataset from `../data/tickets_helpdesk.csv`. Because this notebook is inside the `notebooks` folder, `../data/` means: go one folder back to the project folder, then enter the `data` folder.

The CSV uses tabs between columns, so `sep="\t"` is used to read it correctly.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_PATH = Path("../data/tickets_helpdesk.csv")
OUTPUT_PATH = Path("../data/tickets_helpdesk_clean.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}")

try:
    df = pd.read_csv(DATA_PATH, sep="\t")
except UnicodeDecodeError:
    df = pd.read_csv(DATA_PATH, sep="\t", encoding="latin1")

print(f"Dataset loaded successfully: {df.shape[0]:,} rows and {df.shape[1]:,} columns")
df.head()

Dataset loaded successfully: 20,000 rows and 12 columns


,Ticket_ID,Date_Ouverture,Date_Fermeture,Département,Catégorie,Sous_Catégorie,Priorité,Technicien,Canal,Temps_Resolution_h,SLA,Satisfaction
0,HD000001,2025-12-26 22:41:53.522872,2025-12-27 00:35:10.754447,Direction,Active Directory,Compte verrouillé,Faible,TECH_01,Téléphone,1.89,Respecté,5
1,HD000002,2026-02-10 10:38:33.074503,2026-02-11 06:32:18.905182,Finance,Sécurité,Phishing,Faible,TECH_01,Portail,19.90,Dépassé,1
2,HD000003,2026-03-08 13:06:09.577145,2026-03-08 16:56:27.254580,Production,Messagerie,Outlook,Faible,TECH_09,Email,3.84,Respecté,4
3,HD000004,2026-02-08 18:35:42.139865,2026-02-09 00:04:44.268220,IT,Messagerie,Exchange,Faible,TECH_01,Portail,5.48,Respecté,3
4,HD000005,2025-01-16 05:43:56.864971,2025-01-16 21:48:52.876428,Juridique,Sécurité,Phishing,Faible,TECH_10,Téléphone,16.08,Dépassé,1


## 2. Convert Date Columns to Datetime

`Date_Ouverture` and `Date_Fermeture` are date/time columns. They are converted to datetime format so Python can extract useful information such as the day of the week, hour, and month.

In [2]:
date_columns = ["Date_Ouverture", "Date_Fermeture"]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

pd.DataFrame({
    "column": date_columns,
    "dtype_after_conversion": [df[col].dtype for col in date_columns],
    "invalid_or_missing_dates": [df[col].isna().sum() for col in date_columns]
})

,column,dtype_after_conversion,invalid_or_missing_dates
0,Date_Ouverture,datetime64[us],0
1,Date_Fermeture,datetime64[us],0


## 3. Verify and Handle Missing Values

First, we check missing values in every column.

Chosen strategy:

- Rows with missing critical information are removed because they cannot be trusted for analysis. Critical columns include ticket ID, opening date, closing date, resolution time, SLA, and satisfaction.
- Missing values in non-critical text columns are replaced with `Inconnu`, which means unknown. This keeps the ticket while clearly showing that the information was missing.

In [3]:
missing_before = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

missing_before

,missing_count,missing_percent
Ticket_ID,0,0.00
Date_Ouverture,0,0.00
Date_Fermeture,0,0.00
Département,0,0.00
Catégorie,0,0.00
Sous_Catégorie,0,0.00
Priorité,0,0.00
Technicien,0,0.00
Canal,0,0.00
Temps_Resolution_h,0,0.00


In [5]:
critical_columns = [
    "Ticket_ID",
    "Date_Ouverture",
    "Date_Fermeture",
    "Temps_Resolution_h",
    "SLA",
    "Satisfaction"
]

rows_before_missing_handling = len(df)
df = df.dropna(subset=critical_columns).copy()

text_columns = df.select_dtypes(include=["object", "category"]).columns
non_critical_text_columns = [col for col in text_columns if col not in critical_columns]
df[non_critical_text_columns] = df[non_critical_text_columns].fillna("Inconnu")

rows_removed_missing = rows_before_missing_handling - len(df)

print(f"Rows removed because of missing critical values: {rows_removed_missing:,}")
print(f"Remaining missing values: {df.isna().sum().sum():,}")

pd.DataFrame({
    "missing_count_after": df.isna().sum(),
    "missing_percent_after": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count_after", ascending=False)

Rows removed because of missing critical values: 0
Remaining missing values: 0


C:\Users\HP\AppData\Local\Temp\ipykernel_19628\281833284.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include=["object", "category"]).columns


,missing_count_after,missing_percent_after
Ticket_ID,0,0.00
Date_Ouverture,0,0.00
Date_Fermeture,0,0.00
Département,0,0.00
Catégorie,0,0.00
Sous_Catégorie,0,0.00
Priorité,0,0.00
Technicien,0,0.00
Canal,0,0.00
Temps_Resolution_h,0,0.00


## 4. Remove Duplicate Rows

Duplicate rows can make the same ticket appear more than once in the analysis. We count duplicate rows, remove them if they exist, and verify the result.

In [6]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows before cleaning: {duplicate_count:,}")

df = df.drop_duplicates().copy()

print(f"Duplicate rows after cleaning: {df.duplicated().sum():,}")
print(f"Rows remaining: {len(df):,}")

Duplicate rows before cleaning: 0
Duplicate rows after cleaning: 0
Rows remaining: 20,000


## 5. Verify Resolution Time Values

`Temps_Resolution_h` represents the ticket resolution time in hours. A negative value would not make sense, so we check for negative values and remove them if any exist.

In [7]:
negative_resolution_count = (df["Temps_Resolution_h"] < 0).sum()
print(f"Negative Temps_Resolution_h values: {negative_resolution_count:,}")

if negative_resolution_count > 0:
    df = df[df["Temps_Resolution_h"] >= 0].copy()
    print("Rows with negative resolution time were removed.")
else:
    print("No negative resolution time values found.")

print(f"Rows remaining: {len(df):,}")

Negative Temps_Resolution_h values: 0
No negative resolution time values found.
Rows remaining: 20,000


## 6. Create New Features

We create new columns to support deeper analysis:

- `Jour_Semaine`: the day of the week when the ticket was opened
- `Heure_Ouverture`: the hour when the ticket was opened
- `Mois`: the month when the ticket was opened
- `SLA_Binaire`: converts SLA into numbers, where `Respect? = 1` and `D?pass? = 0`

In [8]:
df["Jour_Semaine"] = df["Date_Ouverture"].dt.day_name()
df["Heure_Ouverture"] = df["Date_Ouverture"].dt.hour
df["Mois"] = df["Date_Ouverture"].dt.month

import unicodedata

def normalize_text(value):
    normalized = unicodedata.normalize("NFKD", str(value))
    return normalized.encode("ascii", "ignore").decode("ascii").lower().strip()

sla_normalized = df["SLA"].apply(normalize_text)
df["SLA_Binaire"] = sla_normalized.map({"respecte": 1, "depasse": 0})

unmapped_sla = df["SLA_Binaire"].isna().sum()
print(f"Rows with unmapped SLA values: {unmapped_sla:,}")

df[["Date_Ouverture", "Jour_Semaine", "Heure_Ouverture", "Mois", "SLA", "SLA_Binaire"]].head(10)

Rows with unmapped SLA values: 20,000


,Date_Ouverture,Jour_Semaine,Heure_Ouverture,Mois,SLA,SLA_Binaire
0,2025-12-26 22:41:53.522872,Friday,22,12,Respecté,NaN
1,2026-02-10 10:38:33.074503,Tuesday,10,2,Dépassé,NaN
2,2026-03-08 13:06:09.577145,Sunday,13,3,Respecté,NaN
3,2026-02-08 18:35:42.139865,Sunday,18,2,Respecté,NaN
4,2025-01-16 05:43:56.864971,Thursday,5,1,Dépassé,NaN
5,2026-03-03 09:09:23.246880,Tuesday,9,3,Respecté,NaN
6,2025-04-27 12:12:01.403560,Sunday,12,4,Dépassé,NaN
7,2025-02-13 18:29:50.490589,Thursday,18,2,Dépassé,NaN
8,2025-05-17 19:57:12.949794,Saturday,19,5,Respecté,NaN
9,2025-10-07 21:24:00.818770,Tuesday,21,10,Respecté,NaN


## 7. Save the Cleaned Dataset

The cleaned dataset is saved as `../data/tickets_helpdesk_clean.csv`. This file will be used for later analysis steps.

In [9]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Cleaned dataset saved to: {OUTPUT_PATH.resolve()}")

Cleaned dataset saved to: C:\helpdesk-ticket-analysis\data\tickets_helpdesk_clean.csv


## 8. Final Verification

Finally, we display the final dataset shape, the first 10 rows, and the data types to confirm that the preparation step worked correctly.

In [10]:
print(f"Final dataset shape: {df.shape[0]:,} rows and {df.shape[1]:,} columns")

Final dataset shape: 20,000 rows and 16 columns


In [11]:
df.head(10)

,Ticket_ID,Date_Ouverture,Date_Fermeture,Département,Catégorie,Sous_Catégorie,Priorité,Technicien,Canal,Temps_Resolution_h,SLA,Satisfaction,Jour_Semaine,Heure_Ouverture,Mois,SLA_Binaire
0,HD000001,2025-12-26 22:41:53.522872,2025-12-27 00:35:10.754447,Direction,Active Directory,Compte verrouillé,Faible,TECH_01,Téléphone,1.89,Respecté,5,Friday,22,12,NaN
1,HD000002,2026-02-10 10:38:33.074503,2026-02-11 06:32:18.905182,Finance,Sécurité,Phishing,Faible,TECH_01,Portail,19.90,Dépassé,1,Tuesday,10,2,NaN
2,HD000003,2026-03-08 13:06:09.577145,2026-03-08 16:56:27.254580,Production,Messagerie,Outlook,Faible,TECH_09,Email,3.84,Respecté,4,Sunday,13,3,NaN
3,HD000004,2026-02-08 18:35:42.139865,2026-02-09 00:04:44.268220,IT,Messagerie,Exchange,Faible,TECH_01,Portail,5.48,Respecté,3,Sunday,18,2,NaN
4,HD000005,2025-01-16 05:43:56.864971,2025-01-16 21:48:52.876428,Juridique,Sécurité,Phishing,Faible,TECH_10,Téléphone,16.08,Dépassé,1,Thursday,5,1,NaN
5,HD000006,2026-03-03 09:09:23.246880,2026-03-03 12:20:25.473536,Direction,Matériel,Imprimante,Faible,TECH_06,Email,3.18,Respecté,4,Tuesday,9,3,NaN
6,HD000007,2025-04-27 12:12:01.403560,2025-04-27 21:03:10.854263,Achats,Sécurité,Antivirus,Moyenne,TECH_09,Téléphone,8.85,Dépassé,4,Sunday,12,4,NaN
7,HD000008,2025-02-13 18:29:50.490589,2025-02-14 03:46:31.601547,IT,Sécurité,Antivirus,Moyenne,TECH_02,Téléphone,9.28,Dépassé,3,Thursday,18,2,NaN
8,HD000009,2025-05-17 19:57:12.949794,2025-05-17 22:21:12.035933,Juridique,Active Directory,Compte verrouillé,Moyenne,TECH_09,Email,2.40,Respecté,5,Saturday,19,5,NaN
9,HD000010,2025-10-07 21:24:00.818770,2025-10-08 01:11:48.531161,RH,Messagerie,Outlook,Faible,TECH_08,Email,3.80,Respecté,5,Tuesday,21,10,NaN


In [12]:
df.dtypes

Ticket_ID                        str
Date_Ouverture        datetime64[us]
Date_Fermeture        datetime64[us]
Département                      str
Catégorie                        str
Sous_Catégorie                   str
Priorité                         str
Technicien                       str
Canal                            str
Temps_Resolution_h           float64
SLA                              str
Satisfaction                   int64
Jour_Semaine                     str
Heure_Ouverture                int32
Mois                           int32
SLA_Binaire                  float64
dtype: object